In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wandb --quiet

## Config

Set `REPO_ROOT` to the repo root on your Google Drive. All other paths are relative to it.

In [ ]:
import os
import sys
import types

import yaml
import torch
import wandb

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
CONFIG_FILE = "config/diffusion_full_celeba_uncond.yaml"
RUN_NAME = "run1"
RUN_NOTES = "Unconditional, CelebA"
RESUME_CKPT_PATH = None

## Imports

In [ ]:
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):

    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from data_utils import build_dataloaders
from models.beta_vae import BetaVAE
from models.unet import UNet
from scripts.diffusion_trainer import DiffusionTrainer
from utils import discover_device, count_model_params

## Load config & build UNet

In [ ]:
from utils import make_run_tag

config_path = os.path.join(REPO_ROOT, CONFIG_FILE)
with open(config_path) as f:
    cfg = types.SimpleNamespace(**yaml.safe_load(f))

cfg.output_dir = os.path.join(REPO_ROOT, f"{cfg.output_dir.lstrip('./')}_{make_run_tag(cfg)}")
cfg.betavae_config_path = os.path.join(REPO_ROOT, cfg.betavae_config_path.lstrip("./"))
cfg.betavae_ckpt_path = os.path.join(REPO_ROOT, cfg.betavae_ckpt_path.lstrip("./"))
if not os.path.isabs(cfg.data_dir):
    cfg.data_dir = os.path.join(REPO_ROOT, cfg.data_dir.lstrip("./"))

if cfg.unconditional:
    print("Train for unconditional generation")

In [ ]:
device = torch.device(discover_device())
print(f"Device: {device}")

model = UNet(
    in_channels=cfg.in_channels,
    image_size=cfg.image_size,
    model_channels=cfg.model_channels,
    channel_mult=cfg.channel_mult,
    num_res_blocks=cfg.num_res_blocks,
    attention_resolutions=cfg.attention_resolutions,
    dropout=cfg.dropout,
    latent_dim=cfg.latent_dim,
)
count_model_params(model)

## Load frozen β-VAE

In [ ]:
if cfg.unconditional:
    vae = torch.nn.Identity()
    print("Unconditional UNet, pass in nn.Identity as VAE placeholder")
else:
    with open(cfg.betavae_config_path) as f:
        vae_cfg = types.SimpleNamespace(**yaml.safe_load(f))

    vae = BetaVAE(
        input_channels=vae_cfg.input_channels,
        input_height=vae_cfg.input_height,
        input_width=vae_cfg.input_width,
        latent_dim=vae_cfg.latent_dim,
        hidden_dim=vae_cfg.hidden_dim,
        beta=vae_cfg.beta,
        device=device,
    )
    ckpt = torch.load(cfg.betavae_ckpt_path, map_location=device)
    vae.load_state_dict(ckpt["model_state_dict"])
    vae.eval()
    print(f"Loaded β-VAE from step {ckpt['step']}")

## Data

In [ ]:
train_dataloader, val_dataloader = build_dataloaders(cfg)

## Wandb

In [ ]:
key="..."
wandb.login(key=key)

wandb.init(
    project="diffusion_ban_map",
    name=f"{cfg.experiment_name}_{RUN_NAME}",
    notes=RUN_NOTES,
    config=vars(cfg),
)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

trainer = DiffusionTrainer(
    model=model,
    vae_model=vae,
    optimizer=optimizer,
    scheduler=None,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    config=cfg,
    device=device,
)
trainer.train(resume_ckpt_path=RESUME_CKPT_PATH)